# CAM · Curso de estrategia
Ejecuta la siguiente celda e introduce la contraseña facilitada en el curso. No escribas la contraseña en una celda de código.

Después ejecuta los ejercicios en orden. El código editable y los notebooks se preparan automáticamente en **Archivos → curso-motorsport**. Haz doble clic en un archivo `.py` para leerlo y editarlo dentro de Colab. No necesitas descargar ni subir nada manualmente. Guarda una copia del notebook en Drive; los cambios en los archivos Python de la sesión son temporales.


In [ ]:
"""This code becomes the first cell of each public Colab notebook."""
import getpass
import hashlib
import io
import json
from pathlib import Path
import subprocess
import sys
import tempfile
import urllib.error
import urllib.request
import zipfile


def preparar_curso():
    endpoint = "https://cam-estrategia-aula.david-simracing14.chatgpt.site/api/materials"
    password = getpass.getpass("Contraseña del curso: ")
    request = urllib.request.Request(
        endpoint, data=json.dumps({"password": password}).encode("utf-8"),
        headers={"Content-Type": "application/json"}, method="POST",
    )
    del password
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            archive = response.read(10 * 1024 * 1024 + 1)
            expected_hash = response.headers.get("X-Course-SHA256")
    except urllib.error.HTTPError as error:
        if error.code == 401:
            raise RuntimeError("Contraseña incorrecta. Vuelve a ejecutar esta celda.") from None
        if error.code == 429:
            raise RuntimeError("Demasiados intentos. Espera 15 minutos y vuelve a probar.") from None
        raise RuntimeError("El acceso al curso no está disponible. Inténtalo más tarde.") from None
    except (urllib.error.URLError, TimeoutError):
        raise RuntimeError("No se pudo conectar. Comprueba la conexión y vuelve a ejecutar la celda.") from None
    finally:
        request.data = None
    if len(archive) > 10 * 1024 * 1024 or not expected_hash or hashlib.sha256(archive).hexdigest() != expected_hash:
        raise RuntimeError("La descarga no es válida. Vuelve a ejecutar la celda.")
    destination = Path.cwd() / "curso-motorsport"
    if destination.exists():
        print("Se conserva tu carpeta curso-motorsport y sus modificaciones.")
    else:
        with zipfile.ZipFile(io.BytesIO(archive)) as package:
            for item in package.infolist():
                target = (Path.cwd() / item.filename).resolve()
                if not target.is_relative_to(destination.resolve()):
                    raise RuntimeError("El archivo contiene una ruta no válida.")
            with tempfile.TemporaryDirectory(dir=Path.cwd()) as temporary:
                package.extractall(temporary)
                (Path(temporary) / "curso-motorsport").rename(destination)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "-e", str(destination)], check=True,
    )
    source = str(destination / "src")
    if source not in sys.path:
        sys.path.insert(0, source)
    print("Acceso correcto. Ya puedes ejecutar los ejercicios.")
    print("Código y notebooks: " + str(destination))


preparar_curso()


# Ejemplo de una carrera

Este cuaderno interactivo (Jupyter) permite inicializar, simular y visualizar una carrera.

## Inicializar

Primero, importaremos las librerías relevantes. En este caso, son todas librerías propias que se pueden encontrar en `Archivos → curso-motorsport → src → race_models → fe`.

In [ ]:
from race_models.fe.config import RaceConfig
from race_models.fe.engine.simulator import simulate_race
from race_models.fe.analysis.race_plots import plot_race

También iniciaremos la configuración de la carrera. En este caso, utilizaremos los valores por defecto.

In [ ]:
race_config = RaceConfig()

In [ ]:
print("Parámetros de la configuración")

print("------------------------------------")

print(f"Número de vueltas: {race_config.total_laps}")
print(f"Número de coches: {race_config.total_drivers}")
print(f"Número de activaciones: {race_config.activation_loss}")
print(f"Posiciones perdidas en activación: {race_config.activation_loss}")
print(f"Adelantamientos por vuelta en AM: {race_config.overtakes_per_lap}")

## Simular la carrera

El código de la siguiente celda simula una carrera según la configuración y retorna el resultado de la carrera.

In [ ]:
result = simulate_race(
    race_config=race_config
)

## Visualizar la carrera

Finalmente, produciremos un gráfico de la carrera para poder analizarla.

In [ ]:
plot_race(result);